# LeetCode 526 — Beautiful Arrangement bằng Python

Link đề: https://leetcode.com/problems/beautiful-arrangement/ · Độ khó: **Medium** · Chủ đề: Bit Manipulation, Array, Dynamic Programming, Backtracking, Bitmask

Skeleton LeetCode:

```python
class Solution:
    def countArrangement(self, n: int) -> int:
        ...
```

Ghi chú: bài chỉ cần số nguyên và list, không có cấu trúc tự dựng. Mọi đáp án trong notebook đều được kiểm chứng bằng cách vét cạn mọi hoán vị với `n` nhỏ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số **hoán vị** của `1..n` sao cho tại **mỗi vị trí `i`** (đánh số từ 1), giá trị `perm[i]` **chia hết cho `i`** hoặc `i` **chia hết cho `perm[i]`**.
2. Nắm các công cụ Python: toán tử `%` để kiểm tra chia hết, `itertools.permutations`, đệ quy quay lui (backtracking) kiểu "chọn → đi tiếp → bỏ chọn", thao tác bit (`1 << k`, `&`, `|`, `bit_count`) và `functools.lru_cache`.
3. Đi từ **thử mọi hoán vị `O(n!)`** → **quay lui cắt nhánh sớm** → **quay lui với danh sách ứng viên tính trước, đi từ vị trí khó nhất** → **quy hoạch động trên bitmask (bottom-up)** → **quy hoạch động bitmask đệ quy có nhớ (top-down)**.
4. Hiểu mấu chốt: khi đã xếp xong `k` vị trí đầu, thứ duy nhất ảnh hưởng tương lai là **tập các số đã dùng**, nên trạng thái chỉ cần là một `mask`; vị trí kế tiếp suy ra được từ số bit `1` trong `mask`.
5. Tránh các lỗi kinh điển: quên bỏ đánh dấu khi quay lui, nhầm vị trí tính từ `0` với tính từ `1`, kiểm tra chia hết một chiều, và nhầm số bit đã bật với vị trí hiện tại (lệch 1).

## 2. Đề bài & ví dụ

Có `n` số nguyên mang nhãn từ `1` đến `n`. Một hoán vị `perm` của các số đó (**đánh số vị trí từ 1**) được gọi là **beautiful arrangement** nếu với mọi `i` (`1 <= i <= n`), **một trong hai** điều kiện sau đúng:

- `perm[i]` chia hết cho `i`;
- `i` chia hết cho `perm[i]`.

Cho số nguyên `n`, hãy trả về **số lượng** beautiful arrangement có thể tạo ra.

**Ví dụ 1**

- Input: `n = 2`
- Output: `2`
- Giải thích: `[1,2]` hợp lệ (`perm[1]=1` chia hết cho 1; `perm[2]=2` chia hết cho 2). `[2,1]` cũng hợp lệ (`perm[1]=2` chia hết cho 1; `i=2` chia hết cho `perm[2]=1`).

**Ví dụ 2**

- Input: `n = 1`
- Output: `1`

**Ràng buộc:**

- `1 <= n <= 15`

Chú ý: `n` chỉ đến `15`, rất nhỏ so với các bài thông thường. Đó là gợi ý mạnh rằng thuật toán có thể **mũ** (như `2^n` hoặc quay lui), nhưng `15! ≈ 1.3 * 10^12` thì vẫn quá lớn để thử mọi hoán vị.

In [1]:
from typing import List
from itertools import permutations
from functools import lru_cache
import random
import time
import math

examples = [(2, 2), (1, 1)]
for n, expected in examples:
    print("n =", n, "-> kỳ vọng", expected)

n = 2 -> kỳ vọng 2
n = 1 -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Vị trí tính từ 1, còn list Python tính từ 0

Đề đánh số vị trí **từ 1**: `perm[1]` là phần tử đầu. Nhưng list Python tính từ **0**: phần tử đầu là `lst[0]`. Khi duyệt một list `p` của Python, vị trí theo đề của `p[k]` là `k + 1`. Đây là nguồn lỗi chính của bài, nên ta sẽ luôn dùng biến tên `pos` cho **vị trí tính từ 1**.

In [2]:
p = [3, 1, 2]               # list Python
for k, v in enumerate(p):
    pos = k + 1             # vị trí theo đề (từ 1)
    print(f"chỉ số list {k} -> vị trí {pos}, giá trị {v}")

chỉ số list 0 -> vị trí 1, giá trị 3
chỉ số list 1 -> vị trí 2, giá trị 1
chỉ số list 2 -> vị trí 3, giá trị 2


### 3.2. Kiểm tra chia hết bằng `%`

`a % b` là **số dư** khi chia `a` cho `b`. `a % b == 0` nghĩa là `a` chia hết cho `b`. Điều kiện của đề: **`v % pos == 0` hoặc `pos % v == 0`** (với `v` là giá trị đặt ở vị trí `pos`). Cả hai chiều đều cần: `v` là bội của `pos`, hoặc `v` là ước của `pos`.

Ví dụ: `(pos=2, v=4)` hợp lệ vì `4 % 2 == 0`; `(pos=4, v=2)` hợp lệ vì `4 % 2 == 0`; `(pos=3, v=2)` không hợp lệ vì `3 % 2 = 1` và `2 % 3 = 2`.

In [3]:
def ok(pos, v):
    """Giá trị v đặt ở vị trí pos (từ 1) có hợp lệ không?"""
    return v % pos == 0 or pos % v == 0

for pos, v in [(2, 4), (4, 2), (3, 2), (1, 5), (6, 6), (4, 3)]:
    print(f"pos={pos}, v={v}: v chia hết pos? {v % pos == 0} | pos chia hết v? {pos % v == 0} -> hợp lệ: {ok(pos, v)}")

pos=2, v=4: v chia hết pos? True | pos chia hết v? False -> hợp lệ: True
pos=4, v=2: v chia hết pos? False | pos chia hết v? True -> hợp lệ: True
pos=3, v=2: v chia hết pos? False | pos chia hết v? False -> hợp lệ: False
pos=1, v=5: v chia hết pos? True | pos chia hết v? False -> hợp lệ: True
pos=6, v=6: v chia hết pos? True | pos chia hết v? True -> hợp lệ: True
pos=4, v=3: v chia hết pos? False | pos chia hết v? False -> hợp lệ: False


### 3.3. `itertools.permutations` và tốc độ tăng của `n!`

`permutations(range(1, n + 1))` sinh **mọi hoán vị** của `1..n`, mỗi hoán vị là một tuple. Số hoán vị là `n! = n * (n-1) * ... * 1`, tăng cực nhanh: `10! = 3.628.800`, `15! ≈ 1.3 * 10^12`.

In [4]:
print(list(permutations([1, 2, 3])))
for n in (3, 5, 8, 10, 12, 15):
    print(f"n = {n:2d}: n! = {math.factorial(n):,}")

[(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]
n =  3: n! = 6
n =  5: n! = 120
n =  8: n! = 40,320
n = 10: n! = 3,628,800
n = 12: n! = 479,001,600
n = 15: n! = 1,307,674,368,000


### 3.4. Quay lui (backtracking): chọn → đi tiếp → bỏ chọn

Quay lui là cách **xây lời giải từng bước**. Ở mỗi bước ta thử từng lựa chọn khả dĩ: (1) **chọn** nó (ghi nhớ đã dùng), (2) **đệ quy** để xây phần còn lại, (3) **bỏ chọn** để thử lựa chọn kế. Bước (3) cực kỳ quan trọng: nếu quên, các nhánh sau sẽ thấy dữ liệu "bẩn" từ nhánh trước.

Ví dụ nhỏ: liệt kê mọi hoán vị của `1..3` bằng quay lui.

In [5]:
def all_perms(n):
    result = []
    used = [False] * (n + 1)
    cur = []
    def dfs():
        if len(cur) == n:
            result.append(cur[:])      # chép lại, vì cur còn thay đổi tiếp
            return
        for v in range(1, n + 1):
            if not used[v]:
                used[v] = True         # chọn
                cur.append(v)
                dfs()                  # đi tiếp
                cur.pop()              # bỏ chọn
                used[v] = False
    dfs()
    return result

print(all_perms(3))

[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


### 3.5. Bitmask: dùng một số nguyên làm "tập hợp"

Với `n <= 15`, ta lưu **tập các số đã dùng** trong **một số nguyên** `mask` có `n` bit: bit thứ `k` (đếm từ 0) bằng `1` nghĩa là số `k + 1` đã được dùng.

- `1 << k`: số có đúng bit `k` bật. Ví dụ `1 << 3 = 8 = 0b1000`.
- `mask & (1 << k)`: khác 0 khi bit `k` đang bật (kiểm tra thuộc tập).
- `mask | (1 << k)`: bật bit `k` (thêm vào tập).
- `mask.bit_count()`: số bit `1` (kích thước tập). Với Python cũ hơn 3.10 dùng `bin(mask).count("1")`.

Số `mask` khác nhau nhiều nhất là `2^n`, nhỏ hơn `n!` rất nhiều (`2^15 = 32768`).

In [6]:
mask = 0b0000
mask |= 1 << 0      # dùng số 1
mask |= 1 << 2      # dùng số 3
print(bin(mask), "| đã dùng số:", [v for v in range(1, 5) if mask & (1 << (v - 1))])
print("số 3 đã dùng?", bool(mask & (1 << 2)), "| số 2 đã dùng?", bool(mask & (1 << 1)))
print("kích thước tập:", mask.bit_count(), "=", bin(mask).count("1"))
print("số mask có thể có với n = 15:", 2 ** 15)

0b101 | đã dùng số: [1, 3]
số 3 đã dùng? True | số 2 đã dùng? False
kích thước tập: 2 = 2
số mask có thể có với n = 15: 32768


### 3.6. `functools.lru_cache`: nhớ kết quả hàm đệ quy

`@lru_cache(None)` đặt trước một hàm sẽ **nhớ** kết quả theo tham số: gọi lại với cùng tham số thì trả về kết quả cũ, không tính lại. Rất hợp để biến đệ quy có "trạng thái lặp lại" thành quy hoạch động top-down. Tham số phải **hashable** (số nguyên thì được).

In [7]:
calls = 0

@lru_cache(None)
def fib(k):
    global calls
    calls += 1
    return k if k < 2 else fib(k - 1) + fib(k - 2)

print(fib(30), "| số lần thân hàm thật sự chạy:", calls)   # 31, không phải hàng triệu

832040 | số lần thân hàm thật sự chạy: 31


## 4. Ý tưởng nền tảng / trực giác

Hãy tính tay với `n = 3`. Với mỗi vị trí `pos`, những giá trị `v` có thể đặt vào đó (thỏa `v % pos == 0` hoặc `pos % v == 0`) là:

| `pos` | các `v` hợp lệ | lý do |
|---|---|---|
| 1 | 1, 2, 3 | mọi số đều chia hết cho 1 |
| 2 | 1, 2 | 1 là ước của 2; 2 chia hết cho 2; 3 thì không |
| 3 | 1, 3 | 1 là ước của 3; 3 chia hết cho 3; 2 thì không |

Bây giờ đếm hoán vị: ta cần chọn `v1 ∈ {1,2,3}`, `v2 ∈ {1,2}`, `v3 ∈ {1,3}` đôi một khác nhau.

- `v3 = 3`: `v2` là `1` hoặc `2`.
  - `v2 = 1` → `v1 = 2`: hoán vị `[2,1,3]`.
  - `v2 = 2` → `v1 = 1`: hoán vị `[1,2,3]`.
- `v3 = 1`: `v2` chỉ còn `2`, rồi `v1 = 3`: hoán vị `[3,2,1]`.

Tổng cộng **3** hoán vị. Thay vì thử cả `3! = 6` hoán vị, ta chỉ xây những cái **còn khả thi**.

In [8]:
n = 3
match = {pos: [v for v in range(1, n + 1) if ok(pos, v)] for pos in range(1, n + 1)}
print(match)

good = [p for p in permutations(range(1, n + 1))
        if all(ok(k + 1, v) for k, v in enumerate(p))]
print("các hoán vị đẹp:", good, "->", len(good))

{1: [1, 2, 3], 2: [1, 2], 3: [1, 3]}
các hoán vị đẹp: [(1, 2, 3), (2, 1, 3), (3, 2, 1)] -> 3


Nhận xét: tại vị trí **lớn**, danh sách ứng viên rất **ngắn** (ví dụ `pos = 13` với `n = 15` chỉ nhận `1` và `13`). Vì vậy nếu ta xếp vị trí khó (ít ứng viên) trước, nhánh nào vô vọng sẽ bị loại ngay.

## 5. Cách 1: Brute Force (thử mọi hoán vị)

**Ý tưởng:** sinh từng hoán vị của `1..n`, kiểm tra điều kiện ở mọi vị trí bằng `all(...)`, đếm hoán vị hợp lệ.

In [9]:
def brute_force(n):
    count = 0
    for p in permutations(range(1, n + 1)):
        if all(v % (k + 1) == 0 or (k + 1) % v == 0 for k, v in enumerate(p)):
            count += 1
    return count

for n, expected in examples + [(3, 3), (4, 8), (5, 10)]:
    got = brute_force(n)
    print("n =", n, "->", got, "| kỳ vọng", expected)
    assert got == expected

n = 2 -> 2 | kỳ vọng 2
n = 1 -> 1 | kỳ vọng 1
n = 3 -> 3 | kỳ vọng 3
n = 4 -> 8 | kỳ vọng 8
n = 5 -> 10 | kỳ vọng 10


**Độ phức tạp:** có `n!` hoán vị, mỗi hoán vị kiểm tra `O(n)`, nên **Time `O(n! * n)`**, Space `O(n)` (một hoán vị tại một thời điểm, không tính bộ nhớ của iterator).

**Nút thắt:** hãy đo thời gian khi `n` tăng:

In [10]:
for n in (6, 7, 8, 9):
    t0 = time.perf_counter()
    r = brute_force(n)
    dt = time.perf_counter() - t0
    print(f"n = {n}: kết quả {r:4d}, mất {dt * 1000:8.1f} ms, số hoán vị {math.factorial(n):,}")
print("n = 15 cần xét", f"{math.factorial(15):,}", "hoán vị -> không thể")

n = 6: kết quả   36, mất      0.3 ms, số hoán vị 720
n = 7: kết quả   41, mất      2.1 ms, số hoán vị 5,040
n = 8: kết quả  132, mất     22.8 ms, số hoán vị 40,320


n = 9: kết quả  250, mất    154.9 ms, số hoán vị 362,880
n = 15 cần xét 1,307,674,368,000 hoán vị -> không thể


Mỗi lần `n` tăng 1, thời gian nhân lên khoảng `n` lần. Ngoài ra, vét cạn **tạo ra cả những hoán vị đã hỏng từ vị trí đầu** rồi mới phát hiện: hoán vị bắt đầu bằng `[2, 3, ...]` đã sai ở vị trí 2 (`3` không đi với `2`), nhưng ta vẫn đi tiếp xét mọi cách xếp phần còn lại.

## 6. Ý tưởng thuật toán / bước đột phá

Có hai nhận xét làm bài từ "không thể" thành "dễ".

**Nhận xét 1 — cắt nhánh sớm (pruning).** Xây hoán vị **từng vị trí một** từ `pos = 1` đến `n`. Tại mỗi vị trí chỉ thử những số **chưa dùng và hợp lệ**. Nếu không còn số nào, nhánh đó chết ngay, không cần xét `(n - pos)!` cách xếp phần sau. Đây là quay lui.

**Nhận xét 2 — trạng thái gọn.** Sau khi đã xếp `k` vị trí đầu, bài toán còn lại là: "xếp các số **chưa dùng** vào các vị trí `k+1, ..., n`". Kết quả chỉ phụ thuộc vào **tập các số đã dùng**, **không** phụ thuộc thứ tự đã xếp chúng. Nhiều cách xếp khác nhau dẫn tới cùng một tập → ta đang giải đi giải lại cùng một bài con.

Nếu mã hóa tập đã dùng bằng `mask`, thì:

- vị trí kế tiếp là `pos = mask.bit_count() + 1` (vì đã dùng `k` số nghĩa là đã xếp `k` vị trí);
- `dp[mask]` = số cách xếp các số trong `mask` vào `k` vị trí đầu sao cho hợp lệ;
- chuyển trạng thái: thử thêm số `v` chưa có trong `mask` vào vị trí `pos` nếu `ok(pos, v)`: `dp[mask | bit(v)] += dp[mask]`.

Chỉ có `2^n` mask nên số trạng thái `2^15 = 32768`, mỗi trạng thái xét `n` lựa chọn: khoảng `5 * 10^5` phép tính. Nhẹ nhàng.

In [11]:
# Số trạng thái mask thật sự "đạt được" (dp[mask] > 0) so với tổng 2^n, và so với n!
def dp_counts(n):
    dp = [0] * (1 << n)
    dp[0] = 1
    for mask in range(1 << n):
        if dp[mask] == 0:
            continue
        pos = mask.bit_count() + 1
        if pos > n:
            continue
        for v in range(1, n + 1):
            bit = 1 << (v - 1)
            if not mask & bit and ok(pos, v):
                dp[mask | bit] += dp[mask]
    reachable = sum(1 for x in dp if x)
    return dp[-1], reachable

for n in (5, 8, 10, 12):
    ans, reach = dp_counts(n)
    print(f"n = {n:2d}: đáp án {ans:5d} | mask đạt được {reach:5d} / 2^n = {2 ** n:6d} | n! = {math.factorial(n):,}")

n =  5: đáp án    10 | mask đạt được    30 / 2^n =     32 | n! = 120
n =  8: đáp án   132 | mask đạt được   214 / 2^n =    256 | n! = 40,320
n = 10: đáp án   700 | mask đạt được   868 / 2^n =   1024 | n! = 3,628,800
n = 12: đáp án  4010 | mask đạt được  3063 / 2^n =   4096 | n! = 479,001,600


## 7. Các cách tiếp theo

Tổng cộng **5 cách**: Cách 1 (đã làm) vét cạn hoán vị; Cách 2 quay lui cắt nhánh sớm; Cách 3 quay lui với danh sách ứng viên và đi từ vị trí khó nhất; Cách 4 quy hoạch động bitmask bottom-up; Cách 5 quy hoạch động bitmask top-down với `lru_cache`.

### 7.1. Cách 2: Quay lui theo vị trí, đánh dấu số đã dùng

**Giải thích từng đoạn:**

1. `used[v]` cho biết số `v` đã được đặt chưa.
2. `dfs(pos)`: đang cần chọn số cho vị trí `pos`. Nếu `pos == n + 1` nghĩa là đã xếp xong `n` vị trí hợp lệ → đếm `1`.
3. Thử từng `v` từ `1..n`: nếu chưa dùng và `ok(pos, v)` thì **chọn** (`used[v] = True`), cộng kết quả của `dfs(pos + 1)`, rồi **bỏ chọn**.
4. Mỗi lời gọi trả về số cách hoàn thành các vị trí còn lại; cộng dồn các nhánh.

In [12]:
def backtrack_used(n):
    used = [False] * (n + 1)

    def dfs(pos):
        if pos == n + 1:
            return 1
        total = 0
        for v in range(1, n + 1):
            if not used[v] and (v % pos == 0 or pos % v == 0):
                used[v] = True              # chọn
                total += dfs(pos + 1)       # đi tiếp
                used[v] = False             # bỏ chọn
        return total

    return dfs(1)

for n, expected in examples + [(3, 3), (4, 8), (5, 10), (6, 36)]:
    got = backtrack_used(n)
    print("n =", n, "->", got)
    assert got == expected

n = 2 -> 2
n = 1 -> 1
n = 3 -> 3
n = 4 -> 8
n = 5 -> 10
n = 6 -> 36


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def countArrangement(self, n: int) -> int:
        used = [False] * (n + 1)

        def dfs(pos):
            if pos == n + 1:
                return 1
            total = 0
            for v in range(1, n + 1):
                if not used[v] and (v % pos == 0 or pos % v == 0):
                    used[v] = True
                    total += dfs(pos + 1)
                    used[v] = False
            return total

        return dfs(1)
```

**Độ phức tạp:** trong trường hợp xấu nhất vẫn là `O(n!)`, nhưng nhờ cắt nhánh nên số nút thực tế nhỏ hơn nhiều (ta sẽ đo ngay dưới). Space `O(n)` cho `used` và ngăn xếp đệ quy sâu `n`.

### 7.2. Cách 3: Quay lui với danh sách ứng viên tính trước, đi từ vị trí `n` về `1`

**Giải thích từng đoạn:**

1. Tính trước `match[pos]` = các `v` thỏa `ok(pos, v)`. Vòng lặp trong `dfs` chỉ duyệt đúng những ứng viên đó, không phải mọi `v` từ `1` đến `n`.
2. **Đi từ vị trí `n` xuống `1`.** Vị trí lớn có ít ứng viên (ví dụ `pos = 13` chỉ nhận `1` và `13`) nên nhánh vô vọng bị cắt sớm; vị trí `1` ở cuối cùng nhận mọi số nên luôn thành công nếu còn đúng một số chưa dùng.
3. `dfs(pos)` trả về số cách xếp các vị trí `pos, pos-1, ..., 1`; điều kiện dừng `pos == 0` trả `1`.

In [13]:
def backtrack_match(n):
    match = [[] for _ in range(n + 1)]
    for pos in range(1, n + 1):
        for v in range(1, n + 1):
            if v % pos == 0 or pos % v == 0:
                match[pos].append(v)
    used = [False] * (n + 1)

    def dfs(pos):
        if pos == 0:
            return 1
        total = 0
        for v in match[pos]:
            if not used[v]:
                used[v] = True
                total += dfs(pos - 1)
                used[v] = False
        return total

    return dfs(n)

for n, expected in examples + [(3, 3), (4, 8), (5, 10), (6, 36)]:
    got = backtrack_match(n)
    print("n =", n, "->", got)
    assert got == expected

n = 2 -> 2
n = 1 -> 1
n = 3 -> 3
n = 4 -> 8
n = 5 -> 10
n = 6 -> 36


Hãy **đếm số nút** đệ quy của Cách 2 so với Cách 3 để thấy lợi ích của việc đi từ vị trí khó và duyệt danh sách ứng viên. Ta chép hai hàm có thêm bộ đếm `nodes` (số lần `dfs` được gọi).

In [14]:
def nodes_forward(n):
    used = [False] * (n + 1)
    nodes = 0
    def dfs(pos):
        nonlocal nodes
        nodes += 1
        if pos == n + 1:
            return 1
        total = 0
        for v in range(1, n + 1):
            if not used[v] and (v % pos == 0 or pos % v == 0):
                used[v] = True
                total += dfs(pos + 1)
                used[v] = False
        return total
    return dfs(1), nodes

def nodes_backward(n):
    match = [[v for v in range(1, n + 1) if v % pos == 0 or pos % v == 0] if pos else [] for pos in range(n + 1)]
    used = [False] * (n + 1)
    nodes = 0
    def dfs(pos):
        nonlocal nodes
        nodes += 1
        if pos == 0:
            return 1
        total = 0
        for v in match[pos]:
            if not used[v]:
                used[v] = True
                total += dfs(pos - 1)
                used[v] = False
        return total
    return dfs(n), nodes

print(f"{'n':>3} {'đáp án':>8} {'nút (đi xuôi)':>15} {'nút (đi ngược)':>16} {'n!':>16}")
for n in (6, 8, 10, 12, 14, 15):
    a1, c1 = nodes_forward(n)
    a2, c2 = nodes_backward(n)
    assert a1 == a2
    print(f"{n:3d} {a1:8d} {c1:15,d} {c2:16,d} {math.factorial(n):16,d}")

  n   đáp án   nút (đi xuôi)   nút (đi ngược)               n!
  6       36             210              124              720
  8      132           1,138              486           40,320
 10      700           8,964            2,700        3,628,800


 12     4010          58,612           15,437      479,001,600


 14    10680         240,066           43,792   87,178,291,200


 15    24679         747,961          102,376 1,307,674,368,000


Hai cột nút đều nhỏ hơn `n!` rất nhiều, và đi ngược cho ít nút hơn đáng kể. Điều đó xác nhận ý tưởng "vị trí khó đi trước".

**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def countArrangement(self, n: int) -> int:
        match = [[] for _ in range(n + 1)]
        for pos in range(1, n + 1):
            for v in range(1, n + 1):
                if v % pos == 0 or pos % v == 0:
                    match[pos].append(v)
        used = [False] * (n + 1)

        def dfs(pos):
            if pos == 0:
                return 1
            total = 0
            for v in match[pos]:
                if not used[v]:
                    used[v] = True
                    total += dfs(pos - 1)
                    used[v] = False
            return total

        return dfs(n)
```

**Độ phức tạp:** Time tệ nhất vẫn `O(n!)` về lý thuyết nhưng nhờ ràng buộc chia hết số nút thực tế rất nhỏ (xem bảng trên: với `n = 15` đi ngược chỉ khoảng `10^5` nút, đi xuôi khoảng `7,5 * 10^5` nút). Space `O(n^2)` cho `match` (tối đa `O(n log n)` thực tế vì mỗi `pos` có không nhiều ứng viên) cộng `O(n)` ngăn xếp.

### 7.3. Cách 4: Quy hoạch động bitmask (bottom-up)

**Giải thích từng đoạn:**

1. `dp[mask]` = số cách xếp **tập số trong `mask`** vào **`k = mask.bit_count()` vị trí đầu** sao cho mọi vị trí hợp lệ.
2. `dp[0] = 1`: không số nào, không vị trí nào, đúng một cách (xếp rỗng).
3. Duyệt `mask` tăng dần (mask nhỏ được xử lý trước mask lớn vì `mask | bit > mask`). Vị trí kế tiếp là `pos = k + 1`. Với mỗi `v` chưa có trong `mask` và `ok(pos, v)`: `dp[mask | bit(v)] += dp[mask]`.
4. Đáp án là `dp[(1 << n) - 1]` (đã dùng đủ `n` số).

In [15]:
def bitmask_dp(n):
    full = (1 << n) - 1
    dp = [0] * (full + 1)
    dp[0] = 1
    for mask in range(full):                 # full chỉ là trạng thái đích, không đi tiếp
        if dp[mask] == 0:
            continue
        pos = mask.bit_count() + 1           # vị trí sắp xếp tiếp theo
        for v in range(1, n + 1):
            bit = 1 << (v - 1)
            if not (mask & bit) and (v % pos == 0 or pos % v == 0):
                dp[mask | bit] += dp[mask]
    return dp[full]

for n, expected in examples + [(3, 3), (4, 8), (5, 10), (6, 36)]:
    got = bitmask_dp(n)
    print("n =", n, "->", got)
    assert got == expected

n = 2 -> 2
n = 1 -> 1
n = 3 -> 3
n = 4 -> 8
n = 5 -> 10
n = 6 -> 36


**Code nộp LeetCode (Cách 4):**

```python
class Solution:
    def countArrangement(self, n: int) -> int:
        full = (1 << n) - 1
        dp = [0] * (full + 1)
        dp[0] = 1
        for mask in range(full):
            if dp[mask] == 0:
                continue
            pos = mask.bit_count() + 1
            for v in range(1, n + 1):
                bit = 1 << (v - 1)
                if not (mask & bit) and (v % pos == 0 or pos % v == 0):
                    dp[mask | bit] += dp[mask]
        return dp[full]
```

**Độ phức tạp:** `2^n` trạng thái, mỗi trạng thái thử `n` giá trị: Time `O(2^n * n)`; Space `O(2^n)` cho mảng `dp`. Với `n = 15`: `32768 * 15 ≈ 4.9 * 10^5` phép tính. Đây là cách có **chặn trên chắc chắn**, không phụ thuộc "may mắn" của cắt nhánh.

### 7.4. Cách 5: Quy hoạch động bitmask top-down (đệ quy có nhớ)

**Ý tưởng:** cùng trạng thái `mask` nhưng viết theo kiểu hỏi: "`f(mask)` là số cách **hoàn thành** các vị trí còn lại khi tập số đã dùng là `mask`". Ở đây ta xếp từ vị trí 1 lên, nên `f(mask)` phụ thuộc `f(mask | bit)`.

**Giải thích từng đoạn:**

1. `pos = mask.bit_count() + 1`. Nếu `pos == n + 1` (mask đầy) trả `1`.
2. Với mỗi `v` chưa dùng, hợp lệ: cộng `f(mask | bit(v))`.
3. `@lru_cache(None)` đảm bảo mỗi `mask` chỉ được tính một lần.

Khác với Cách 4, ta chỉ đi qua những mask **thật sự đạt được**, không quét mọi `2^n` mask.

In [16]:
def bitmask_memo(n):
    full = (1 << n) - 1

    @lru_cache(None)
    def f(mask):
        if mask == full:
            return 1
        pos = mask.bit_count() + 1
        total = 0
        for v in range(1, n + 1):
            bit = 1 << (v - 1)
            if not (mask & bit) and (v % pos == 0 or pos % v == 0):
                total += f(mask | bit)
        return total

    return f(0)

for n, expected in examples + [(3, 3), (4, 8), (5, 10), (6, 36)]:
    got = bitmask_memo(n)
    print("n =", n, "->", got)
    assert got == expected

n = 2 -> 2
n = 1 -> 1
n = 3 -> 3
n = 4 -> 8
n = 5 -> 10
n = 6 -> 36


**Code nộp LeetCode (Cách 5):**

```python
class Solution:
    def countArrangement(self, n: int) -> int:
        full = (1 << n) - 1

        @cache
        def f(mask):
            if mask == full:
                return 1
            pos = mask.bit_count() + 1
            total = 0
            for v in range(1, n + 1):
                bit = 1 << (v - 1)
                if not (mask & bit) and (v % pos == 0 or pos % v == 0):
                    total += f(mask | bit)
            return total

        return f(0)
```

(Trên LeetCode, `cache` hoặc `lru_cache(None)` đều dùng được; chú ý `n` đổi giữa các test nên hàm `f` phải được định nghĩa **bên trong** `countArrangement`, như trên, để không dính cache của test trước.)

**Độ phức tạp:** Time `O(2^n * n)`, Space `O(2^n)` cho cache cộng ngăn xếp đệ quy sâu `n`.

## 8. TRACE từng bước

### 8.1. Trace quay lui (Cách 2) với `n = 3`

Mỗi dòng in ra cho biết: độ sâu (thụt lề), vị trí đang xét, số được chọn, và khi nào bỏ chọn. Khi `pos == n + 1` ta tìm thấy một hoán vị đẹp. Với `n = 3` ta kỳ vọng thấy đúng 3 hoán vị: `[1,2,3]`, `[2,1,3]`, `[3,2,1]`.

In [17]:
def trace_backtrack(n):
    used = [False] * (n + 1)
    cur = []
    found = 0

    def dfs(pos):
        nonlocal found
        indent = "    " * (pos - 1)
        if pos == n + 1:
            found += 1
            print(f"{indent}=> tìm thấy hoán vị #{found}: {cur}")
            return
        print(f"{indent}vị trí {pos}: ứng viên chưa dùng và hợp lệ = "
              f"{[v for v in range(1, n + 1) if not used[v] and ok(pos, v)]}")
        for v in range(1, n + 1):
            if used[v]:
                continue
            if not ok(pos, v):
                print(f"{indent}  loại v={v} (không chia hết với pos={pos})")
                continue
            print(f"{indent}  chọn v={v}")
            used[v] = True
            cur.append(v)
            dfs(pos + 1)
            cur.pop()
            used[v] = False
            print(f"{indent}  bỏ chọn v={v}")
    dfs(1)
    print("TỔNG:", found)
    return found

trace_backtrack(3)

vị trí 1: ứng viên chưa dùng và hợp lệ = [1, 2, 3]
  chọn v=1
    vị trí 2: ứng viên chưa dùng và hợp lệ = [2]
      chọn v=2
        vị trí 3: ứng viên chưa dùng và hợp lệ = [3]
          chọn v=3
            => tìm thấy hoán vị #1: [1, 2, 3]
          bỏ chọn v=3
      bỏ chọn v=2
      loại v=3 (không chia hết với pos=2)
  bỏ chọn v=1
  chọn v=2
    vị trí 2: ứng viên chưa dùng và hợp lệ = [1]
      chọn v=1
        vị trí 3: ứng viên chưa dùng và hợp lệ = [3]
          chọn v=3
            => tìm thấy hoán vị #2: [2, 1, 3]
          bỏ chọn v=3
      bỏ chọn v=1
      loại v=3 (không chia hết với pos=2)
  bỏ chọn v=2
  chọn v=3
    vị trí 2: ứng viên chưa dùng và hợp lệ = [1, 2]
      chọn v=1
        vị trí 3: ứng viên chưa dùng và hợp lệ = []
          loại v=2 (không chia hết với pos=3)
      bỏ chọn v=1
      chọn v=2
        vị trí 3: ứng viên chưa dùng và hợp lệ = [1]
          chọn v=1
            => tìm thấy hoán vị #3: [3, 2, 1]
          bỏ chọn v=1
      bỏ chọn v=2
  bỏ

3

Quan sát nhánh `chọn v=2` ở vị trí 1: sang vị trí 2 chỉ còn `1` hợp lệ (vì `2` đã dùng, `3` không hợp lệ); sang vị trí 3 chỉ còn `3`. Nhánh `chọn v=3` ở vị trí 1: vị trí 2 chỉ nhận `1` hoặc `2`; chọn `1` thì vị trí 3 cần `1` hoặc `3`, cả hai đều đã dùng → nhánh chết, đó là sự cắt nhánh.

### 8.2. Trace quy hoạch động bitmask (Cách 4) với `n = 3`

Bảng `dp` (mask viết nhị phân 3 bit; bit phải nhất là số `1`):

| `mask` | tập đã dùng | `pos` kế tiếp | chuyển đi các mask | giá trị `dp[mask]` |
|---|---|---|---|---|
| `000` | `{}` | 1 | `001`, `010`, `100` đều +1 | 1 |
| `001` | `{1}` | 2 | `011` (thêm 2) +1 | 1 |
| `010` | `{2}` | 2 | `011` (thêm 1) +1 | 1 |
| `011` | `{1,2}` | 3 | `111` (thêm 3) +2 | 2 |
| `100` | `{3}` | 2 | `101` (thêm 1) +1, `110` (thêm 2) +1 | 1 |
| `101` | `{1,3}` | 3 | thêm 2: không hợp lệ (3 không đi với 2) | 1 |
| `110` | `{2,3}` | 3 | `111` (thêm 1) +1 | 1 |
| `111` | đủ | | đáp án | 3 |

In [18]:
def trace_dp(n):
    full = (1 << n) - 1
    dp = [0] * (full + 1)
    dp[0] = 1
    for mask in range(full):
        if dp[mask] == 0:
            print(f"mask={mask:0{n}b}: dp=0 -> bỏ qua")
            continue
        pos = mask.bit_count() + 1
        used = [v for v in range(1, n + 1) if mask >> (v - 1) & 1]
        print(f"mask={mask:0{n}b}: đã dùng {used}, dp={dp[mask]}, xếp vị trí {pos}")
        for v in range(1, n + 1):
            bit = 1 << (v - 1)
            if mask & bit:
                continue
            if ok(pos, v):
                dp[mask | bit] += dp[mask]
                print(f"    thêm v={v}: dp[{(mask | bit):0{n}b}] += {dp[mask]}  -> {dp[mask | bit]}")
            else:
                print(f"    thêm v={v}: không hợp lệ ({v} và {pos} không chia hết nhau)")
    print("ĐÁP ÁN dp[full] =", dp[full])
    return dp[full]

trace_dp(3)

mask=000: đã dùng [], dp=1, xếp vị trí 1
    thêm v=1: dp[001] += 1  -> 1
    thêm v=2: dp[010] += 1  -> 1
    thêm v=3: dp[100] += 1  -> 1
mask=001: đã dùng [1], dp=1, xếp vị trí 2
    thêm v=2: dp[011] += 1  -> 1
    thêm v=3: không hợp lệ (3 và 2 không chia hết nhau)
mask=010: đã dùng [2], dp=1, xếp vị trí 2
    thêm v=1: dp[011] += 1  -> 2
    thêm v=3: không hợp lệ (3 và 2 không chia hết nhau)
mask=011: đã dùng [1, 2], dp=2, xếp vị trí 3
    thêm v=3: dp[111] += 2  -> 2
mask=100: đã dùng [3], dp=1, xếp vị trí 2
    thêm v=1: dp[101] += 1  -> 1
    thêm v=2: dp[110] += 1  -> 1
mask=101: đã dùng [1, 3], dp=1, xếp vị trí 3
    thêm v=2: không hợp lệ (2 và 3 không chia hết nhau)
mask=110: đã dùng [2, 3], dp=1, xếp vị trí 3
    thêm v=1: dp[111] += 1  -> 3
ĐÁP ÁN dp[full] = 3


3

Chú ý ở dòng `mask=101`: vị trí kế tiếp là 3 và số còn lại là `2`; `3 % 2 = 1` và `2 % 3 = 2` nên không hợp lệ, vì vậy `dp[111]` không nhận thêm từ nhánh này. Đó chính là lý do `dp[101] = 1` nhưng không đóng góp vào đáp án.

Thử thêm trace `n = 4` (đáp án 8) nhưng chỉ in kết quả cuối để so với kết quả đã tính tay ở bài tập.

In [19]:
for n in (4, 5):
    print("n =", n, ": quay lui =", backtrack_used(n), "| DP =", bitmask_dp(n), "| memo =", bitmask_memo(n))

n = 4 : quay lui = 8 | DP = 8 | memo = 8
n = 5 : quay lui = 10 | DP = 10 | memo = 10


## 9. Code nộp LeetCode

Ta nộp **Cách 4 (DP bitmask)** vì có chặn trên rõ ràng `O(2^n * n)` và ngắn gọn.

In [20]:
class Solution:
    def countArrangement(self, n: int) -> int:
        full = (1 << n) - 1
        dp = [0] * (full + 1)
        dp[0] = 1                              # xếp rỗng: 1 cách
        for mask in range(full):
            if dp[mask] == 0:
                continue
            pos = mask.bit_count() + 1         # vị trí kế tiếp (tính từ 1)
            for v in range(1, n + 1):
                bit = 1 << (v - 1)
                if not (mask & bit) and (v % pos == 0 or pos % v == 0):
                    dp[mask | bit] += dp[mask]
        return dp[full]


sol = Solution()
for n, expected in examples:
    got = sol.countArrangement(n)
    print("n =", n, "->", got, "| kỳ vọng", expected)
    assert got == expected
print("n = 15 ->", sol.countArrangement(15))

n = 2 -> 2 | kỳ vọng 2
n = 1 -> 1 | kỳ vọng 1
n = 15 -> 24679


## 10. Vì sao đúng

**Mô hình hóa.** Một beautiful arrangement tương ứng với một dãy lựa chọn `v1, v2, ..., vn` đôi một khác nhau (một hoán vị của `1..n`) và `ok(i, vi)` đúng với mọi `i`. Thuật toán xây dãy từng phần tử theo thứ tự `i = 1, 2, ..., n`.

**Tính đầy đủ và không trùng.** Mỗi hoán vị đẹp ứng với **đúng một** đường đi trong cây quyết định: tại bước `i` ta chọn giá trị `vi` của nó. Hai hoán vị khác nhau rẽ khác nhau ở bước đầu tiên chúng khác nhau. Nhánh bị cắt (không có ứng viên) không chứa hoán vị đẹp nào nên bỏ đi không làm mất đáp án.

**Vì sao DP trên `mask` đúng.** Giả sử đã xếp `k` vị trí đầu bằng tập số `S` (`|S| = k`). Số cách hoàn thành `k+1..n` chỉ phụ thuộc `S` (các số còn lại là `{1..n} \ S` và các vị trí còn lại cố định), không phụ thuộc thứ tự đã xếp `S`. Vậy ta có thể gộp mọi cách xếp `S` vào một ô `dp[S]` — cộng số cách. Vị trí kế tiếp xác định bởi `|S| + 1`, nên không cần lưu thêm.

**Kiểm chứng bằng code:** (a) liệt kê mọi hoán vị đẹp bằng quay lui rồi kiểm tra từng cái thực sự thỏa điều kiện, đôi một khác nhau, đúng là hoán vị của `1..n`; (b) số lượng khớp với DP.

In [21]:
def list_arrangements(n):
    used = [False] * (n + 1)
    cur, out = [], []
    def dfs(pos):
        if pos == n + 1:
            out.append(tuple(cur))
            return
        for v in range(1, n + 1):
            if not used[v] and ok(pos, v):
                used[v] = True; cur.append(v)
                dfs(pos + 1)
                cur.pop(); used[v] = False
    dfs(1)
    return out

for n in range(1, 9):
    arrs = list_arrangements(n)
    assert len(set(arrs)) == len(arrs)                                    # đôi một khác nhau
    for a in arrs:
        assert sorted(a) == list(range(1, n + 1))                         # là hoán vị của 1..n
        assert all(a[k] % (k + 1) == 0 or (k + 1) % a[k] == 0 for k in range(n))   # đẹp
    assert len(arrs) == bitmask_dp(n) == brute_force(n)
    print(f"n = {n}: {len(arrs):4d} hoán vị đẹp, đều hợp lệ và đầy đủ")

n = 1:    1 hoán vị đẹp, đều hợp lệ và đầy đủ
n = 2:    2 hoán vị đẹp, đều hợp lệ và đầy đủ
n = 3:    3 hoán vị đẹp, đều hợp lệ và đầy đủ
n = 4:    8 hoán vị đẹp, đều hợp lệ và đầy đủ
n = 5:   10 hoán vị đẹp, đều hợp lệ và đầy đủ
n = 6:   36 hoán vị đẹp, đều hợp lệ và đầy đủ
n = 7:   41 hoán vị đẹp, đều hợp lệ và đầy đủ
n = 8:  132 hoán vị đẹp, đều hợp lệ và đầy đủ


## 11. Time/Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn `permutations` | `O(n! * n)` | `O(n)` | chỉ dùng được với `n <= 9` |
| 2. Quay lui `used[]` | `O(n!)` tệ nhất, thực tế nhỏ hơn nhiều | `O(n)` | cắt nhánh sớm |
| 3. Quay lui + `match` + đi ngược | `O(n!)` tệ nhất, thực tế nhỏ nhất | `O(n^2)` | ít nút nhất |
| 4. DP bitmask bottom-up | `O(2^n * n)` | `O(2^n)` | chặn trên chắc chắn |
| 5. DP bitmask top-down | `O(2^n * n)` | `O(2^n)` | chỉ đi qua mask đạt được |

**Với bài này (`n <= 15`):** `2^15 * 15 = 491.520` phép tính cho DP. Quay lui đi ngược cũng chỉ khoảng `10^5` nút (đo ở mục 7.2). Cả hai đều chạy trong chưa tới một giây.

**Vì sao không phải `O(n!)` chặt chẽ?** Quay lui thử rất ít nhánh vì ràng buộc chia hết loại bỏ nhiều giá trị. Nhưng khó chứng minh chặn trên tốt, nên DP bitmask với `O(2^n * n)` là lựa chọn an toàn khi viết lời giải có chứng minh.

## 12. Bộ test

Ta kiểm tra: ví dụ của đề, edge case, bảng đáp án đã biết cho mọi `n` từ `1` đến `15`, phản ví dụ cho các cách làm sai, đối chiếu mọi cách giải với oracle vét cạn (cho `n` nhỏ), đối chiếu giữa các cách nhanh với `n` lớn (chọn ngẫu nhiên có seed), và kiểm tra thời gian.

In [22]:
# 12.1 ví dụ của đề và edge case (n nhỏ nhất, n lớn nhất)
sol = Solution()
for n, expected in [(2, 2), (1, 1), (3, 3), (4, 8), (5, 10), (6, 36), (7, 41), (8, 132)]:
    assert sol.countArrangement(n) == expected, n
print("n = 1, 2 và các giá trị nhỏ: OK")

# 12.2 bảng đáp án cho mọi n trong ràng buộc: so sánh DP, quay lui, memo
table = [1, 2, 3, 8, 10, 36, 41, 132, 250, 700, 750, 4010, 4237, 10680, 24679]
for n in range(1, 16):
    a = Solution().countArrangement(n)
    b = backtrack_match(n)
    c = bitmask_memo(n)
    assert a == b == c == table[n - 1], (n, a, b, c)
print("n = 1..15: DP, quay lui (đi ngược), memo đều khớp:", table)

n = 1, 2 và các giá trị nhỏ: OK


n = 1..15: DP, quay lui (đi ngược), memo đều khớp: [1, 2, 3, 8, 10, 36, 41, 132, 250, 700, 750, 4010, 4237, 10680, 24679]


In [23]:
# 12.3 đối chiếu với oracle vét cạn độc lập (hoán vị bằng itertools) cho n <= 8
def oracle(n):
    total = 0
    for p in permutations(range(1, n + 1)):
        good = True
        for k in range(n):
            pos, v = k + 1, p[k]
            if v % pos != 0 and pos % v != 0:
                good = False
                break
        total += good
    return total

methods = {
    "brute_force": brute_force,
    "backtrack_used": backtrack_used,
    "backtrack_match": backtrack_match,
    "bitmask_dp": bitmask_dp,
    "bitmask_memo": bitmask_memo,
    "Solution": Solution().countArrangement,
}
for n in range(1, 9):
    expected = oracle(n)
    for name, fn in methods.items():
        assert fn(n) == expected, (name, n)
print("n = 1..8: cả 6 cách khớp oracle")

# 12.4 ngẫu nhiên có seed: các cách nhanh phải khớp nhau với n tới 15
rng = random.Random(526)
fast = ["backtrack_used", "backtrack_match", "bitmask_dp", "bitmask_memo", "Solution"]
for _ in range(25):
    n = rng.randint(1, 15)
    results = {name: methods[name](n) for name in fast}
    assert len(set(results.values())) == 1, (n, results)
    assert results["Solution"] == table[n - 1]
print("25 giá trị n ngẫu nhiên (1..15): các cách nhanh khớp nhau và khớp bảng")

n = 1..8: cả 6 cách khớp oracle


25 giá trị n ngẫu nhiên (1..15): các cách nhanh khớp nhau và khớp bảng


In [24]:
# 12.5 phản ví dụ: các cách làm SAI phổ biến
def wrong_no_undo(n):                    # quên bỏ chọn
    used = [False] * (n + 1)
    def dfs(pos):
        if pos == n + 1:
            return 1
        total = 0
        for v in range(1, n + 1):
            if not used[v] and ok(pos, v):
                used[v] = True
                total += dfs(pos + 1)
                # THIẾU used[v] = False
        return total
    return dfs(1)

def wrong_one_way(n):                    # chỉ kiểm tra v chia hết pos
    used = [False] * (n + 1)
    def dfs(pos):
        if pos == n + 1:
            return 1
        total = 0
        for v in range(1, n + 1):
            if not used[v] and v % pos == 0:
                used[v] = True
                total += dfs(pos + 1)
                used[v] = False
        return total
    return dfs(1)

def wrong_off_by_one(n):                 # DP nhưng pos = bit_count() (lệch 1)
    full = (1 << n) - 1
    dp = [0] * (full + 1)
    dp[0] = 1
    for mask in range(full):
        pos = mask.bit_count()           # SAI: thiếu + 1  -> pos có thể là 0
        for v in range(1, n + 1):
            bit = 1 << (v - 1)
            if not mask & bit and pos and (v % pos == 0 or pos % v == 0):
                dp[mask | bit] += dp[mask]
    return dp[full]

for n in (3, 4, 5):
    print(f"n={n}: đúng={table[n - 1]:3d} | không bỏ chọn={wrong_no_undo(n):3d} | "
          f"một chiều={wrong_one_way(n):3d} | lệch 1={wrong_off_by_one(n):3d}")
assert wrong_no_undo(4) != 8 and wrong_one_way(4) != 8 and wrong_off_by_one(4) != 8
print("các cách sai bị phát hiện")

n=3: đúng=  3 | không bỏ chọn=  1 | một chiều=  1 | lệch 1=  0
n=4: đúng=  8 | không bỏ chọn=  1 | một chiều=  1 | lệch 1=  0
n=5: đúng= 10 | không bỏ chọn=  1 | một chiều=  1 | lệch 1=  0
các cách sai bị phát hiện


In [25]:
# 12.6 thời gian: n = 15 (lớn nhất theo ràng buộc)
for name in ("backtrack_used", "backtrack_match", "bitmask_dp", "bitmask_memo"):
    t0 = time.perf_counter()
    r = methods[name](15)
    dt = time.perf_counter() - t0
    assert r == 24679
    print(f"{name:16s} n=15 -> {r}  ({dt * 1000:7.1f} ms)")

backtrack_used   n=15 -> 24679  (  489.2 ms)
backtrack_match  n=15 -> 24679  (   23.5 ms)
bitmask_dp       n=15 -> 24679  (   25.5 ms)
bitmask_memo     n=15 -> 24679  (   28.4 ms)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên bỏ chọn (`used[v] = False`)

Sau khi đệ quy xong ta phải trả `used[v]` về trạng thái cũ, nếu không các nhánh sau thấy số `v` "đã bị dùng" dù thực tế không.

In [26]:
def buggy_1(n):
    used = [False] * (n + 1)
    def dfs(pos):
        if pos == n + 1:
            return 1
        total = 0
        for v in range(1, n + 1):
            if not used[v] and (v % pos == 0 or pos % v == 0):
                used[v] = True
                total += dfs(pos + 1)
                # SAI: quên used[v] = False
        return total
    return dfs(1)

print("SAI  n=3:", buggy_1(3), "(đúng 3)")
print("SAI  n=4:", buggy_1(4), "(đúng 8)")
print("ĐÚNG n=4:", backtrack_used(4))

SAI  n=3: 1 (đúng 3)
SAI  n=4: 1 (đúng 8)
ĐÚNG n=4: 8


**Vì sao sai:** sau nhánh đầu tiên, mọi số đã đánh dấu mãi mãi, nên các nhánh sau không còn gì để chọn. **Sửa:** thêm `used[v] = False` ngay sau lời gọi đệ quy.

### Lỗi 2: Nhầm vị trí tính từ 0 với tính từ 1

Nếu duyệt `for k in range(n)` rồi dùng `k` làm vị trí, `k = 0` sẽ gây `ZeroDivisionError` (chia cho `0`), còn nếu bỏ qua `k = 0` thì các vị trí còn lại lệch 1 nên đáp án sai.

In [27]:
def buggy_2(n):
    # SAI: dùng chỉ số 0-based làm "vị trí"
    count = 0
    for p in permutations(range(1, n + 1)):
        good = True
        for k, v in enumerate(p):
            if k == 0:
                continue                 # né ZeroDivisionError, nhưng bỏ sót ràng buộc
            if v % k != 0 and k % v != 0:
                good = False
                break
        count += good
    return count

print("SAI  n=4:", buggy_2(4), "(đúng 8)")
print("ĐÚNG n=4:", brute_force(4))
try:
    5 % 0
except ZeroDivisionError as e:
    print("nếu không né vị trí 0:", type(e).__name__, "-", e)

SAI  n=4: 10 (đúng 8)
ĐÚNG n=4: 8
nếu không né vị trí 0: ZeroDivisionError - integer modulo by zero


**Vì sao sai:** vị trí `k` ở list Python tương ứng vị trí `k + 1` của đề. **Sửa:** luôn đặt `pos = k + 1` hoặc cho vòng lặp chạy `pos` từ `1`.

### Lỗi 3: Kiểm tra chia hết một chiều

Đề cho phép **cả hai** chiều: `perm[i] % i == 0` **hoặc** `i % perm[i] == 0`. Chỉ kiểm tra một chiều loại mất các hoán vị hợp lệ (ví dụ `[2,1]` ở `n = 2`).

In [28]:
def buggy_3(n):
    used = [False] * (n + 1)
    def dfs(pos):
        if pos == n + 1:
            return 1
        total = 0
        for v in range(1, n + 1):
            if not used[v] and v % pos == 0:        # SAI: thiếu "or pos % v == 0"
                used[v] = True
                total += dfs(pos + 1)
                used[v] = False
        return total
    return dfs(1)

print("SAI  n=2:", buggy_3(2), "(đúng 2)")
print("SAI  n=3:", buggy_3(3), "(đúng 3)")
print("ĐÚNG n=3:", backtrack_used(3))

SAI  n=2: 1 (đúng 2)
SAI  n=3: 1 (đúng 3)
ĐÚNG n=3: 3


**Vì sao sai:** chỉ một chiều thì chỉ còn hoán vị đơn vị `[1,2,...,n]` (mỗi số đứng đúng chỗ nó chia hết). **Sửa:** `(v % pos == 0 or pos % v == 0)`.

### Lỗi 4: Lệch 1 giữa số bit đã bật và vị trí hiện tại (DP bitmask)

Nếu đã dùng `k` số thì ta đã xếp `k` vị trí, vị trí **kế tiếp** là `k + 1`. Dùng `pos = k` thì `pos = 0` ở trạng thái rỗng (chia cho 0) và mọi vị trí lệch.

In [29]:
print("SAI  n=4 (lệch 1):", wrong_off_by_one(4), "(đúng 8)")
print("ĐÚNG n=4         :", bitmask_dp(4))

SAI  n=4 (lệch 1): 0 (đúng 8)
ĐÚNG n=4         : 8


**Vì sao sai:** với `pos = bit_count()`, trạng thái rỗng có `pos = 0` nên không có chuyển đi nào, `dp` dừng ở `0`. **Sửa:** `pos = mask.bit_count() + 1`.

## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra bài dạng này:**

- Ràng buộc `n` **rất nhỏ** (`n <= 15` hoặc `n <= 20`) và yêu cầu đếm/tối ưu trên **hoán vị** hoặc **tập con** → nghĩ ngay tới **quay lui** hoặc **DP bitmask**.
- Điều kiện chỉ phụ thuộc "**phần tử nào đã dùng**" chứ không phụ thuộc thứ tự dùng → trạng thái gọn là `mask`.
- Có ràng buộc "phần tử này không hợp với vị trí kia" → danh sách ứng viên cho mỗi vị trí.

**Mẹo:**

1. **Xếp vị trí ít lựa chọn trước** (most-constrained-first) để cắt nhánh sớm. Ở bài này là vị trí lớn.
2. Tính trước danh sách ứng viên cho từng vị trí thay vì lặp lại phép chia nhiều lần.
3. **Quay lui xong là chạy DP bitmask được ngay** khi trạng thái chỉ là tập đã dùng: vị trí hiện tại = `popcount(mask) + 1`.
4. Luôn kiểm tra lỗi lệch 1: vị trí tính từ 1, chỉ số list tính từ 0, bit thứ `v - 1` ứng với số `v`.
5. Nếu hai kiểu trạng thái cho cùng kết quả (như `used[]` và `mask`), `mask` cho phép **nhớ kết quả** (memo/DP), còn `used[]` thì không.

## 15. Biến thể

### 15.1. Liệt kê (không chỉ đếm) các hoán vị đẹp

Đã làm ở mục 10 với hàm `list_arrangements`. Chú ý: khi cần **liệt kê** thì DP đếm không giúp được nữa (đáp án có thể rất lớn); phải dùng quay lui và chấp nhận chi phí bằng số kết quả.

In [30]:
print(list_arrangements(4))
print("số lượng:", len(list_arrangements(4)))

[(1, 2, 3, 4), (1, 4, 3, 2), (2, 1, 3, 4), (2, 4, 3, 1), (3, 2, 1, 4), (3, 4, 1, 2), (4, 1, 3, 2), (4, 2, 3, 1)]
số lượng: 8


### 15.2. Bài tổng quát: đếm hoán vị thỏa một luật tùy ý

Ta đổi `ok(pos, v)` thành tham số. Cùng một khung DP bitmask dùng được cho nhiều bài: đổi luật là đổi bài.

In [31]:
def count_with_rule(n, rule):
    """Đếm hoán vị của 1..n sao cho rule(pos, v) đúng với mọi vị trí pos (từ 1)."""
    full = (1 << n) - 1
    dp = [0] * (full + 1)
    dp[0] = 1
    for mask in range(full):
        if not dp[mask]:
            continue
        pos = mask.bit_count() + 1
        for v in range(1, n + 1):
            bit = 1 << (v - 1)
            if not mask & bit and rule(pos, v):
                dp[mask | bit] += dp[mask]
    return dp[full]

beautiful = lambda pos, v: v % pos == 0 or pos % v == 0
print([count_with_rule(n, beautiful) for n in range(1, 9)])          # bài gốc
print([count_with_rule(n, lambda pos, v: pos != v) for n in range(1, 8)])   # mất thứ tự (derangement)
print([count_with_rule(n, lambda pos, v: v % pos == 0) for n in range(1, 8)])   # chỉ một chiều

[1, 2, 3, 8, 10, 36, 41, 132]
[0, 1, 2, 9, 44, 265, 1854]
[1, 1, 1, 1, 1, 1, 1]


- Luật `pos != v` cho số **hoán vị không có điểm cố định (derangement)**: `0, 1, 2, 9, 44, 265, 1854`.
- Luật `v % pos == 0` (một chiều) cho `1` hoán vị với mọi `n` — vì vị trí `n` chỉ nhận `n`, rồi `n - 1`, ...

### 15.3. Bảng đáp án có sẵn (tra bảng)

Vì `n` chỉ có 15 giá trị, ta có thể **tính trước** và dán bảng vào code. Nó nhanh nhất về thời gian chạy, nhưng đó là "nộp đáp án" chứ không giải thuật toán; trong phỏng vấn người ta sẽ hỏi cách tìm ra bảng. Ta dùng bảng này để kiểm tra lẫn nhau ở mục 12, không dùng để nộp.

### 15.4. Các bài cùng họ

- **LeetCode 667 — Beautiful Arrangement II:** bài xây dựng (không đếm), dùng ý tưởng xen kẽ đầu–cuối.
- **LeetCode 47 — Permutations II, 46 — Permutations:** khung quay lui hoán vị.
- **LeetCode 996 — Number of Squareful Arrays:** đếm hoán vị mà tổng hai phần tử kề là số chính phương; DP bitmask hoặc quay lui có tập ứng viên.
- **LeetCode 698 — Partition to K Equal Sum Subsets, 1879 — Minimum XOR Sum of Two Arrays:** DP bitmask với `n` nhỏ.
- **LeetCode 526 mở rộng:** đổi luật chia hết bằng một luật khác (như `count_with_rule` ở trên).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn `permutations` | sinh mọi hoán vị rồi lọc | `O(n! * n)` | `O(n)` | `n <= 9`, dùng làm oracle |
| 2. Quay lui `used[]` | xây theo vị trí, cắt khi hết ứng viên | `O(n!)` tệ nhất, thực tế nhỏ | `O(n)` | cách dễ viết nhất, đủ qua bài |
| 3. Quay lui + `match`, đi ngược | duyệt danh sách ứng viên, vị trí khó trước | `O(n!)` tệ nhất, ít nút nhất | `O(n^2)` | khi muốn quay lui nhanh nhất |
| 4. DP bitmask bottom-up | `dp[mask]`, `pos = popcount + 1` | `O(2^n * n)` | `O(2^n)` | chặn trên chắc chắn; **nên nộp** |
| 5. DP bitmask top-down | `f(mask)` đệ quy có nhớ | `O(2^n * n)` | `O(2^n)` | khi trạng thái đạt được thưa |

Kết luận: để học thì bắt đầu từ Cách 2 (hiểu quay lui), rồi nhận ra "trạng thái chỉ là tập đã dùng" để chuyển sang Cách 4/5. Cách 3 minh họa nguyên tắc cắt nhánh quan trọng.

## 17. Bài tập vận dụng

Làm lần lượt từ dễ đến khó. Đáp án ở **mục 19**. Các cell TODO **chưa được gọi**; bạn tự điền rồi tự chạy.

### Bài 1 (đoán output)

Không chạy code, hãy đoán in ra gì:

```python
n = 4
match = {pos: [v for v in range(1, n + 1) if v % pos == 0 or pos % v == 0]
         for pos in range(1, n + 1)}
print(match[2], len(match[3]), match[4])
```

### Bài 2 (tính tay)

Với `n = 4`, liệt kê danh sách ứng viên cho từng vị trí rồi **đếm tay** số hoán vị đẹp (gợi ý: chia trường hợp theo giá trị của vị trí 3). Đáp án là bao nhiêu?

### Bài 3 (điền TODO)

Điền các chỗ trống của DP bitmask:

In [32]:
def ex3_fill(n):
    full = (1 << n) - 1
    dp = [0] * (full + 1)
    dp[0] = 1
    for mask in range(full):
        if dp[mask] == 0:
            continue
        pos = ...                       # TODO 1: vị trí kế tiếp (từ 1) theo số bit đã bật
        for v in range(1, n + 1):
            bit = ...                   # TODO 2: bit đại diện cho số v
            if not (mask & bit) and ...:    # TODO 3: điều kiện chia hết hai chiều
                dp[mask | bit] += dp[mask]
    return dp[full]

# Khi đã điền xong, bỏ dấu # ở dòng sau để thử:
# assert ex3_fill(4) == 8

### Bài 4 (tự viết lại)

Viết lại **Cách 5 (DP bitmask top-down với `lru_cache`)** từ đầu, không nhìn bài giảng. Chú ý định nghĩa hàm đệ quy **bên trong** hàm chính.

In [33]:
def ex4_memo(n):
    # TODO: tự viết hàm f(mask) có @lru_cache và trả về f(0)
    ...

# Khi đã viết xong, bỏ dấu # để thử:
# assert ex4_memo(5) == 10

### Bài 5 (mini challenge, biến thể)

Viết hàm `count_with_rule(n, rule)` đếm hoán vị của `1..n` thỏa `rule(pos, v)` ở mọi vị trí `pos` (từ 1). Rồi dùng nó để tính:

1. luật "một chiều" `v % pos == 0` cho `n = 1..8`;
2. luật "ngược" `pos % v == 0` cho `n = 1..8`;
3. luật "đẹp nhưng **vị trí 1 phải chứa `n`**" cho `n = 1..8` (gợi ý: bọc rule gốc).

Đối chiếu từng kết quả với vét cạn `permutations`.

In [34]:
def ex5_count_with_rule(n, rule):
    # TODO: DP bitmask tổng quát theo luật rule(pos, v)
    ...

# Khi đã viết xong, bỏ dấu # để thử:
# assert ex5_count_with_rule(4, lambda pos, v: v % pos == 0) == 1

## 18. Gợi ý 3 mức

**Bài 3**

- Mức 1 (nhẹ): đã dùng `k` số thì đã xếp mấy vị trí? Vị trí kế tiếp là vị trí thứ mấy?
- Mức 2 (vừa): số `v` ứng với bit thứ `v - 1`; dùng phép dịch bit `1 << ...`.
- Mức 3 (gần đáp án): `pos = mask.bit_count() + 1`; `bit = 1 << (v - 1)`; điều kiện `(v % pos == 0 or pos % v == 0)`.

**Bài 4**

- Mức 1: `full = (1 << n) - 1` là mask khi đã dùng đủ `n` số.
- Mức 2: khi `mask == full` trả `1`; ngược lại cộng `f(mask | bit)` cho mọi `v` hợp lệ chưa dùng.
- Mức 3: đặt `@lru_cache(None)` ngay trên `def f(mask):` bên trong `ex4_memo`, cuối cùng `return f(0)`.

**Bài 5**

- Mức 1: gần như chép lại `bitmask_dp`, thay điều kiện chia hết bằng `rule(pos, v)`.
- Mức 2: luật số 3 có thể viết `lambda pos, v: beautiful(pos, v) and (pos != 1 or v == n)`.
- Mức 3: vét cạn để so sánh: `sum(all(rule(k + 1, p[k]) for k in range(n)) for p in permutations(range(1, n + 1)))`.

## 19. Đáp án tham khảo

**Bài 1:** `match = {1: [1,2,3,4], 2: [1,2,4], 3: [1,3], 4: [1,2,4]}`. In ra: `[1, 2, 4] 2 [1, 2, 4]`.

**Bài 2:** ứng viên: `pos1: {1,2,3,4}`, `pos2: {1,2,4}`, `pos3: {1,3}`, `pos4: {1,2,4}`.

- Vị trí 3 là `3`: `pos2` và `pos4` lấy hai số khác nhau từ `{1,2,4}` có thứ tự → `3 * 2 = 6` cách, số còn lại về `pos1` (hợp lệ vì mọi số đi với 1) → **6** hoán vị.
- Vị trí 3 là `1`: `pos2`, `pos4` lấy từ `{2,4}` (vì `1` đã dùng) có thứ tự → `2` cách, `pos1` nhận `3` → **2** hoán vị.

Tổng **8**, khớp với đáp án `n = 4`.

In [35]:
# Bài 1
n = 4
match = {pos: [v for v in range(1, n + 1) if v % pos == 0 or pos % v == 0] for pos in range(1, n + 1)}
assert match == {1: [1, 2, 3, 4], 2: [1, 2, 4], 3: [1, 3], 4: [1, 2, 4]}
print("Bài 1:", match[2], len(match[3]), match[4])

# Bài 2: đếm tay theo trường hợp bằng code
cases = 0
for v3 in match[3]:
    rest = [v for v in (1, 2, 3, 4) if v != v3]
    for v2 in match[2]:
        for v4 in match[4]:
            used = {v3, v2, v4}
            if len(used) == 3:
                v1 = next(v for v in (1, 2, 3, 4) if v not in used)
                if v1 in match[1]:
                    cases += 1
assert cases == 8 == brute_force(4)
print("Bài 2:", cases)

# Bài 3
def ex3_answer(n):
    full = (1 << n) - 1
    dp = [0] * (full + 1)
    dp[0] = 1
    for mask in range(full):
        if dp[mask] == 0:
            continue
        pos = mask.bit_count() + 1
        for v in range(1, n + 1):
            bit = 1 << (v - 1)
            if not (mask & bit) and (v % pos == 0 or pos % v == 0):
                dp[mask | bit] += dp[mask]
    return dp[full]

# Bài 4
def ex4_answer(n):
    full = (1 << n) - 1

    @lru_cache(None)
    def f(mask):
        if mask == full:
            return 1
        pos = mask.bit_count() + 1
        total = 0
        for v in range(1, n + 1):
            bit = 1 << (v - 1)
            if not (mask & bit) and (v % pos == 0 or pos % v == 0):
                total += f(mask | bit)
        return total

    return f(0)

# Bài 5
def ex5_answer(n, rule):
    full = (1 << n) - 1
    dp = [0] * (full + 1)
    dp[0] = 1
    for mask in range(full):
        if not dp[mask]:
            continue
        pos = mask.bit_count() + 1
        for v in range(1, n + 1):
            bit = 1 << (v - 1)
            if not mask & bit and rule(pos, v):
                dp[mask | bit] += dp[mask]
    return dp[full]

def ex5_oracle(n, rule):
    return sum(all(rule(k + 1, p[k]) for k in range(n)) for p in permutations(range(1, n + 1)))

for n in range(1, 10):
    assert ex3_answer(n) == ex4_answer(n) == table[n - 1]
print("Bài 3, 4: khớp bảng đáp án với n = 1..9")

for n in range(1, 9):
    r1 = lambda pos, v: v % pos == 0
    r2 = lambda pos, v: pos % v == 0
    r3 = lambda pos, v, n=n: (v % pos == 0 or pos % v == 0) and (pos != 1 or v == n)
    for rule in (r1, r2, r3):
        assert ex5_answer(n, rule) == ex5_oracle(n, rule), (n,)
print("Bài 5: luật một chiều      ", [ex5_answer(n, lambda pos, v: v % pos == 0) for n in range(1, 9)])
print("Bài 5: luật ngược          ", [ex5_answer(n, lambda pos, v: pos % v == 0) for n in range(1, 9)])
print("Bài 5: vị trí 1 chứa n     ", [ex5_answer(n, lambda pos, v, n=n: (v % pos == 0 or pos % v == 0) and (pos != 1 or v == n)) for n in range(1, 9)])

Bài 1: [1, 2, 4] 2 [1, 2, 4]
Bài 2: 8
Bài 3, 4: khớp bảng đáp án với n = 1..9


Bài 5: luật một chiều       [1, 1, 1, 1, 1, 1, 1, 1]
Bài 5: luật ngược           [1, 1, 1, 1, 1, 1, 1, 1]
Bài 5: vị trí 1 chứa n      [1, 1, 1, 2, 2, 6, 5, 18]


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template DP bitmask đếm hoán vị với ràng buộc theo vị trí:**

```python
class Solution:
    def countArrangement(self, n: int) -> int:
        full = (1 << n) - 1
        dp = [0] * (full + 1)
        dp[0] = 1                                   # xếp rỗng
        for mask in range(full):
            if dp[mask] == 0:
                continue
            pos = mask.bit_count() + 1              # vị trí kế tiếp, tính từ 1
            for v in range(1, n + 1):
                bit = 1 << (v - 1)
                if not (mask & bit) and (v % pos == 0 or pos % v == 0):
                    dp[mask | bit] += dp[mask]
        return dp[full]
```

**Template quay lui (nếu thích):** `used[]`, `dfs(pos)`, chọn → đi tiếp → bỏ chọn; điều kiện dừng `pos == n + 1`.

**Checklist trước khi submit:**

- [ ] Vị trí tính từ `1` (`pos = bit_count() + 1`), số `v` ứng với bit `v - 1`?
- [ ] Điều kiện chia hết kiểm tra **cả hai chiều** (`or`)?
- [ ] `dp[0] = 1`, đáp án là `dp[(1 << n) - 1]`?
- [ ] Nếu quay lui: đã **bỏ chọn** sau khi đệ quy?
- [ ] Nếu dùng `lru_cache`: hàm đệ quy nằm **trong** `countArrangement` để cache không dính giữa các test?
- [ ] Đã thử `n = 1`, `n = 2`, và `n = 15` (kết quả `24679`)?
- [ ] Độ phức tạp `O(2^n * n)` với `n <= 15`?

**Ghi nhớ một câu:** *"Chỉ cần biết những số nào đã dùng; vị trí kế tiếp = số số đã dùng + 1."*